- $\pi_\theta(a|s)$
- old_log_prob: $\pi_{\text{old}}(a|s)$
- ref_log_prob: $\pi_{\text{ref}}(a|s)$

### steps

- trainer.total_epochs=15
- data.train_batch_size=128
    - rollout.n=4: 128 * 4 = 512, rollout batch size
    - total steps: 2101 / 128 * 15 = 16 * 15 = 240 steps
- PPO Epochs (ppo_epochs=1)：
    - 日志中默认 PPO Epoch 为 1，意味着这 512 条数据只会被用于更新 Actor 模型 1 次。
- actor_rollout_ref.actor.ppo_mini_batch_size=32 (prompts 级别的)
    - 更新时不会一次性将 512 (128*4) 条数据塞入模型，而是将其切分为若干个 Mini-Batch。
    - 设定为 32 意味着每次参数优化（Optimizer Step）会计算 32 条 prompts 的梯度。
        - 因此，在一个 Global Step (1/240) 中，会进行 128/32=4 次模型参数更新（Optimizer Steps）。
    - PPO 的“prompt 级”mini-batch。注意 FSDP worker 会按 rollout.n 和 world size (8卡节点) 重新归一化，
        - 32*4 / 8 = 16  (prompt & responses)
- actor_rollout_ref.actor.ppo_micro_batch_size_per_gpu=2 (response级别)
    - 每一张显卡单次前向/反向传播处理 2 条数据。(8卡节点) => global micro-batch size: 8 * 2 = 16
    - Gradient Accumulation Steps = ppo_mini_batch_size (32) * 4 / Global Micro-Batch Size (16) = 8。
        - 系统会进行 8 次前向+反向传播来累加梯度，然后执行 1 次 Optimizer 权重更新。

- 脚本输入：
    - actor.ppo_mini_batch_size = 32（这里你写的是 32个 Prompt）
    - rollout.n = 4（每个 Prompt 4个 Response）
    - actor.ppo_micro_batch_size_per_gpu = 2（限制单卡每次只能塞 2个 Response）
    - n_gpus_per_node = 8（8卡DP）
- 框架内部真实执行的数据切分逻辑：
    - 全局总数据：512 条 Responses (128 prompts * 4)
    - 全局切分为 Mini-Batch，每次处理 128 条 Responses (32 prompts * 4)。
    - 分发到 8 张卡，单卡分到的 Mini-Batch 大小为 16 条 Responses。
    - 因为单卡显存只能塞下 micro=2 条 Responses，所以单卡需要做 16/2=8 次的前向和反向计算。
        - 做完 8 次累计后（梯度累加步数为 8），执行一次 Optimizer Step 更新参数。

### fsdp_config

```
    actor_rollout_ref.actor.fsdp_config.param_offload=False \
    actor_rollout_ref.actor.fsdp_config.optimizer_offload=False \
```

- 这里配置的 offload 选项，实际上是传给 PyTorch 底层 `FSDP(cpu_offload=CPUOffload(...))` 的（对应你如果在代码里追踪，它决定了 self._is_offload_param 等内部变量）。
    - 在“训练（Update/Backward）阶段”内部。 如果把它设为 True，FSDP 会在做前向传播（Forward）和反向传播（Backward）时，算完一层（Layer）就把这一层的参数和优化器立刻卸载回 CPU。这能极大节省显存，但会让训练速度慢得令人发指（因为 PCIe 通信太频繁了）。 所以在实际的 GPU 训练脚本中，由于 GPU 显存（比如 A100 80G）勉强够用，我们通常把这里的 FSDP 动态 Offload 设为 False，以保证纯计算阶段的最佳性能。
    - 这里跟 phase 切换时的（training => rollout），`self.actor.engine.to("cpu", model=True, optimizer=False, grad=False)`
        - 跨越不同训练生命周期（Train vs. Rollout）的资源调度。 不管底层 FSDP 的配置参数怎么写，一旦进入生成阶段，由于需要庞大的 KV Cache，框架必须强制将模型权重踢到 CPU 去。

```
    actor_rollout_ref.ref.fsdp_config.param_offload=True \
```


### rollout